# Asistente Turístico de Tenerife 2026

Este notebook demuestra el funcionamiento del asistente turístico desarrollado para la guía de Tenerife.  
Incluye:

- Recuperación de información desde el PDF ***(RAG)***
- Uso de herramientas externas (tiempo real, transporte, restaurantes)
- Memoria conversacional multiturno
- Generación de respuestas con el modelo de OpenAI

El objetivo es mostrar ejemplos prácticos de uso y validar que el sistema funciona correctamente.

Este notebook está diseñado para ejecutarse tras preparar el entorno con make setup, que crea el entorno virtual, instala dependencias y registra el kernel Tenerife Assistant.

```bash
make setup


## Importación de módulos del asistente

En esta sección cargamos la clase principal `TenerifeAssistant`, que encapsula:

- RAG (FAISS + embeddings)
- Memoria conversacional
- Function calling
- Herramientas externas (clima, transporte, restaurantes)

In [1]:
import os, sys

ROOT = os.path.abspath(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("ROOT:", ROOT)


ROOT: /home/alexd/GitHub/tenerife-assistant-2026


In [2]:
from src.assistant import TenerifeAssistant
import inspect
inspect.signature(TenerifeAssistant)


<Signature (pdf_path: str, model: str = 'gpt-4.1-mini', temperature: float = 0.2, top_p: float = 1.0, max_tokens: int = 500)>

## Carga del asistente turístico

En esta celda inicializamos la clase `Tenerife Assistant 2026`, que realiza:

- Carga del PDF de la guía turística
- División del documento en chunks
- Generación de embeddings
- Construcción o carga del índice FAISS
- Inicialización de la memoria conversacional
- Configuración del modelo LLM y de las herramientas externas

La primera ejecución puede tardar unos segundos mientras se construye el índice vectorial.


In [3]:
# Inicialización del asistente turístico

assistant = TenerifeAssistant(
    pdf_path="data/TENERIFE.pdf",
    model="gpt-4.1-mini",          # Modelo comercial usado
    temperature=0.2,               # Parámetros expuestos (requisito del enunciado)
    top_p=1.0,
    max_tokens=500
)

assistant


2026-06-11 14:48:38,883 - TenerifeAssistant - INFO - Cargando pipeline RAG desde data/TENERIFE.pdf
2026-06-11 14:48:38,884 - src.rag - INFO - Iniciando pipeline RAG para: data/TENERIFE.pdf
2026-06-11 14:48:38,885 - src.rag - INFO - Cargando PDF desde: data/TENERIFE.pdf


DEBUG MODEL: gpt-4.1-mini


2026-06-11 14:48:39,508 - src.rag - INFO - PDF cargado: 25 páginas
2026-06-11 14:48:39,509 - src.rag - INFO - Segmentando texto: chunk_size=800, overlap=150
2026-06-11 14:48:39,511 - src.rag - INFO - Texto segmentado en 35 chunks con metadatos
2026-06-11 14:48:39,520 - src.rag - INFO - Generando embeddings para 35 chunks...
2026-06-11 14:48:41,421 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 14:48:41,439 - src.rag - INFO - Embeddings generados: forma (35, 1536)
2026-06-11 14:48:41,441 - src.rag - INFO - Construyendo índice FAISS...
2026-06-11 14:48:41,444 - src.rag - INFO - Índice FAISS creado con 35 vectores
2026-06-11 14:48:41,445 - src.rag - INFO - Pipeline RAG completado exitosamente
2026-06-11 14:48:41,446 - src.memory - INFO - ConversationMemory inicializada: max_tokens=2500, model=gpt-4.1-mini
2026-06-11 14:48:41,446 - TenerifeAssistant - INFO - TenerifeAssistant listo para responder preguntas.


## Prueba de RAG (recuperación de información)

En esta sección comprobamos que el asistente es capaz de:

- Recuperar información relevante de la guía
- Citar la fuente utilizada
- Responder de forma coherente a preguntas sobre Tenerife

In [4]:
pregunta_rag = "¿Qué playas tranquilas recomiendas en Tenerife para familias?"
respuesta_rag = assistant.answer(pregunta_rag)
print("PREGUNTA:", pregunta_rag)
print("\nRESPUESTA:\n", respuesta_rag)

2026-06-11 15:20:24,264 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué playas tranquilas recomiendas en Tenerife par'
2026-06-11 15:20:24,265 - src.rag - INFO - Buscando 4 chunks relevantes para: '¿Qué playas tranquilas recomiendas en Tenerife par...'
2026-06-11 15:20:25,769 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:20:25,774 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:20:28,520 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:28,550 - TenerifeAssistant - INFO - Respuesta generada en 4.29s


PREGUNTA: ¿Qué playas tranquilas recomiendas en Tenerife para familias?

RESPUESTA:
 Para familias que buscan playas tranquilas en Tenerife, una buena opción es visitar la Playa del Camisón en Costa Adeje. Esta playa es pequeña y suele ser bastante agradable para un ambiente familiar [4].

Otra opción puede ser la Playa de Las Vistas en Los Cristianos, que es conocida por ser una playa apta para familias y con buenas instalaciones [1].

En general, las playas del sur de Tenerife tienden a ser más de arena blanca y tranquilas, ideales para familias, a diferencia del norte donde predominan las playas de arena negra [4][1].

Si necesitas información sobre servicios o restaurantes en estas zonas, puedo ayudarte también.

**Fuentes:**
[1] Guía turística (pág. 23)
[2] Guía turística (pág. 11)
[3] Guía turística (pág. 16)
[4] Guía turística (pág. 20)


## Prueba de function calling: predicción del tiempo

Uno de los puntos es tener al menos una función externa, por ejemplo:

#### - `get_weather(fecha)`

En esta sección probamos varias consultas relacionadas con el tiempo, incluyendo:

- Casos correctos
- Manejo de errores
- Registro de intentos en el log (si está implementado en `TenerifeAssistant`)

In [5]:
consultas_tiempo = [
    "¿Qué tiempo hace hoy en Tenerife?",
    "¿Qué tiempo hará mañana en Santa Cruz de Tenerife?",
    "¿Va a llover este fin de semana en el norte de la isla?",
]

for q in consultas_tiempo:
    print("=" * 80)
    print("PREGUNTA:", q)
    r = assistant.answer(q)
    print("\nRESPUESTA:\n", r)


2026-06-11 15:20:38,892 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué tiempo hace hoy en Tenerife?'


PREGUNTA: ¿Qué tiempo hace hoy en Tenerife?


2026-06-11 15:20:39,887 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:39,891 - TenerifeAssistant - INFO - Llamada a herramienta: get_weather args={'location': 'Tenerife'}
2026-06-11 15:20:39,892 - src.tools - INFO - Consultando tiempo para Santa Cruz de Tenerife
2026-06-11 15:20:41,797 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:41,800 - TenerifeAssistant - INFO - Respuesta generada en 2.91s
2026-06-11 15:20:41,801 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué tiempo hará mañana en Santa Cruz de Tenerife?'



RESPUESTA:
 Hoy en Tenerife, específicamente en Santa Cruz de Tenerife, el tiempo es de cielo claro con una temperatura agradable de aproximadamente 27.6 °C. La humedad está en un 77% y hay viento con una velocidad de 21.8 km/h. Es un día ideal para disfrutar al aire libre y en la playa.
PREGUNTA: ¿Qué tiempo hará mañana en Santa Cruz de Tenerife?


2026-06-11 15:20:42,973 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:42,978 - TenerifeAssistant - INFO - Llamada a herramienta: get_weather args={'location': 'Santa Cruz de Tenerife', 'date': '2024-06-12'}
2026-06-11 15:20:42,979 - src.tools - INFO - Consultando tiempo para Santa Cruz de Tenerife
2026-06-11 15:20:44,559 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:44,563 - TenerifeAssistant - INFO - Respuesta generada en 2.76s
2026-06-11 15:20:44,564 - TenerifeAssistant - INFO - Consulta recibida: '¿Va a llover este fin de semana en el norte de la '
2026-06-11 15:20:44,564 - src.rag - INFO - Buscando 4 chunks relevantes para: '¿Va a llover este fin de semana en el norte de la ...'



RESPUESTA:
 Mañana en Santa Cruz de Tenerife se espera un día de cielo claro con una temperatura de aproximadamente 27.6 °C. La humedad será del 77% y el viento soplará a 21.8 km/h. Es un buen día para actividades al aire libre y disfrutar del clima agradable.
PREGUNTA: ¿Va a llover este fin de semana en el norte de la isla?


2026-06-11 15:20:45,712 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:20:45,714 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:20:47,132 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:20:47,136 - TenerifeAssistant - INFO - Llamada a herramienta: get_weather args={'location': 'Puerto de La Cruz', 'date': '2024-06-15'}
2026-06-11 15:20:47,137 - src.tools - INFO - Consultando tiempo para Puerto de La Cruz
2026-06-11 15:20:47,274 - src.rag - INFO - Buscando 4 chunks relevantes para: 'Usa el siguiente resultado para responder al usuar...'
2026-06-11 15:20:47,431 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:20:47,434 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:20:48,696 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



RESPUESTA:
 Este fin de semana en el norte de Tenerife, específicamente en Puerto de La Cruz, no se espera lluvia. El clima será de cielo claro con temperaturas alrededor de 22.4 °C, humedad del 77% y vientos suaves de aproximadamente 9.6 km/h. Perfecto para salir y disfrutar de la zona.


### Prueba herramienta mock: ***guaguas***

In [6]:
assistant.answer("¿Qué paradas de guagua hay cerca de La Laguna?")

2026-06-11 15:22:44,690 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué paradas de guagua hay cerca de La Laguna?'
2026-06-11 15:22:45,626 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:22:45,628 - TenerifeAssistant - INFO - Llamada a herramienta: get_bus_stops args={'location': 'La Laguna'}
2026-06-11 15:22:46,754 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:22:46,760 - TenerifeAssistant - INFO - Respuesta generada en 2.07s


'Cerca de La Laguna, las paradas de guagua disponibles son: Aguere, Padre Anchieta, La Trinidad, San Agustín y Campamento Alto. Estas paradas facilitan el acceso a diferentes rutas y conexiones en la zona.'

### Prueba herramienta mock: ***restaurantes***

In [7]:
assistant.answer("¿Hay alguna oferta gastronómica interesante en Adeje?")

2026-06-11 15:22:52,092 - TenerifeAssistant - INFO - Consulta recibida: '¿Hay alguna oferta gastronómica interesante en Ade'
2026-06-11 15:22:52,743 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:22:52,752 - TenerifeAssistant - INFO - Llamada a herramienta: get_restaurant_offers args={'location': 'Adeje'}
2026-06-11 15:22:54,075 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:22:54,083 - TenerifeAssistant - INFO - Respuesta generada en 1.99s


'En Adeje hay ofertas gastronómicas interesantes para aprovechar. Por ejemplo, en el restaurante "El Gomero" ofrecen un postre gratis y se especializan en cocina canaria. También en "La Cueva" puedes disfrutar de un 2x1 en tapas en un ambiente acogedor. Son opciones perfectas para probar la gastronomía local con descuentos atractivos.'

## Prueba de memoria conversacional multiturno

En esta sección comprobamos que el asistente:

- Mantiene el contexto entre turnos
- Recuerda preferencias del usuario
- Puede referirse a información mencionada anteriormente

In [8]:
turnos = [
    "Quiero organizar un viaje de 5 días a Tenerife.",
    "Me gustan las rutas de senderismo y las playas tranquilas.",
    "Prefiero evitar sitios demasiado turísticos.",
    "¿Me puedes proponer un plan para esos 5 días?",
    "¿Puedes recordarme qué tipo de actividades te dije que me gustaban?",
]

for q in turnos:
    print("=" * 80)
    print("USUARIO:", q)
    r = assistant.answer(q)
    print("\nASISTENTE:\n", r)


2026-06-11 15:22:59,452 - TenerifeAssistant - INFO - Consulta recibida: 'Quiero organizar un viaje de 5 días a Tenerife.'
2026-06-11 15:22:59,453 - src.rag - INFO - Buscando 4 chunks relevantes para: 'Quiero organizar un viaje de 5 días a Tenerife....'


USUARIO: Quiero organizar un viaje de 5 días a Tenerife.


2026-06-11 15:22:59,712 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:22:59,714 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:23:05,328 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:05,333 - TenerifeAssistant - INFO - Respuesta generada en 5.88s
2026-06-11 15:23:05,334 - TenerifeAssistant - INFO - Consulta recibida: 'Me gustan las rutas de senderismo y las playas tra'



ASISTENTE:
 Para organizar un viaje de 5 días a Tenerife, te sugiero el siguiente plan para aprovechar al máximo la isla:

Día 1: Explora Santa Cruz de Tenerife  
- Aparca en el aparcamiento del Parque Marítimo.  
- Camina por la Avenida Marítima hasta Plaza de España.  
- Visita el Auditorio de Tenerife en el camino.  
- Callejea por Calle Castillo, Plaza Weyler, Parque García Sanabria, y Plaza del Príncipe.  
- Acaba el día en la playa de Las Teresitas.

Día 2: Parque Nacional del Teide  
- Sube por la carretera TF24 (carretera de La Esperanza).  
- Para en el Mirador de La Tarta para disfrutar del paisaje.  
- Usa el teleférico para subir al pico del Teide.  
- Visita el Centro de Visitantes de El Portillo.  
- Si el cielo está despejado, considera subir de noche para ver el cielo estrellado.

Día 3: Costa Adeje y playas  
- Disfruta de las playas de arena blanca como Playa de Torviscas y Playa del Camisón.  
- Explora la zona turística con hoteles de lujo, parques acuáticos, pubs,

2026-06-11 15:23:10,987 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:10,995 - TenerifeAssistant - INFO - Respuesta generada en 5.66s
2026-06-11 15:23:10,996 - TenerifeAssistant - INFO - Consulta recibida: 'Prefiero evitar sitios demasiado turísticos.'
2026-06-11 15:23:10,997 - src.rag - INFO - Buscando 4 chunks relevantes para: 'Prefiero evitar sitios demasiado turísticos....'


...


ASISTENTE:
 Si prefieres evitar sitios demasiado turísticos en Tenerife y disfrutar de senderismo y playas tranquilas, te recomiendo este plan más exclusivo y alejado de las multitudes:

Día 1: Ruta de senderismo en el Barranco de Masca  
- Visita el pueblo de Masca, un lugar pintoresco y menos masificado.  
- Realiza la ruta del Barranco de Masca, que ofrece paisajes espectaculares. Es recomendable reservar con antelación y comprobar las restricciones actuales [2].  
- Disfruta del mirador de Masca si no quieres hacer toda la ruta.

Día 2: Playas tranquilas en Buenavista del Norte  
- Explora playas menos concurridas cerca de la zona de Buenavista.  
- Aparca cerca de la Plaza de los Remedios y disfruta de la zona y un dulce en la pastelería El Aderno [1].

Día 3: Senderismo en Anaga  
- Realiza rutas en el Parque Rural de Anaga, especialmente desde miradores como La Jardina, considerado uno de los favoritos [4].  
- Busca senderos menos conocidos para evitar aglomeraciones.

Día 4: 

2026-06-11 15:23:29,090 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:34,833 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:34,840 - src.memory - INFO - Historial recortado (eliminados roles: ['user', 'assistant']).
2026-06-11 15:23:34,844 - src.memory - INFO - Historial recortado (eliminados roles: ['user', 'assistant']).
2026-06-11 15:23:34,848 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'assistant']).
2026-06-11 15:23:34,850 - TenerifeAssistant - INFO - Respuesta generada en 17.90s
2026-06-11 15:23:34,851 - TenerifeAssistant - INFO - Consulta recibida: '¿Puedes recordarme qué tipo de actividades te dije'
2026-06-11 15:23:34,852 - src.rag - INFO - Buscando 4 chunks relevantes para: '¿Puedes recordarme qué tipo de actividades te dije...'



ASISTENTE:
 Claro, aquí tienes un plan detallado para 5 días en Tenerife que combina senderismo y playas tranquilas, evitando las zonas masificadas:

Día 1: Senderismo en Barranco de Masca
- Mañana: Llegada al pintoresco pueblo de Masca. Paseo por el pueblo y admirar las casas tradicionales.  
- Tarde: Realizar la ruta de senderismo por el Barranco de Masca (asesórate sobre el estado del sendero y las condiciones climáticas). La ruta ofrece paisajes espectaculares y naturaleza virgen.  

Día 2: Playas tranquilas en Buenavista del Norte  
- Día entero para relajarte en playas poco turísticas del noroeste de Tenerife, como Playa de la Arena o Playa de El Socorro. Son ideales para disfrutar del mar en un entorno más natural y sereno.  

Día 3: Senderismo en Anaga  
- Mañana y tarde: Explora el Parque Rural de Anaga, centrando la ruta en zonas menos concurridas como La Jardina. Podrás disfrutar de bosques de laurisilva y miradores maravillosos sin aglomeraciones.  

Día 4: Playas menos ac

2026-06-11 15:23:35,142 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:23:35,144 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:23:35,147 - TenerifeAssistant - INFO - Contexto supera 2500 tokens (3084). Se crea resumen del historial.
2026-06-11 15:23:40,147 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:41,096 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:23:41,106 - src.memory - INFO - Historial recortado (eliminados roles: ['user']).
2026-06-11 15:23:41,111 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'assistant']).
2026-06-11 15:23:41,114 - TenerifeAssistant - INFO - Respuesta generada en 6.26s



ASISTENTE:
 Me dijiste que te gustan las actividades de senderismo y disfrutar de playas tranquilas, evitando sitios demasiado turísticos.

**Fuentes:**
[1] Guía turística (pág. 21)
[2] Guía turística (pág. 24)
[3] Guía turística (pág. 1)
[4] Guía turística (pág. 24)


## Ejemplo de conversación larga y/o combinada

A continuación se muestra una conversación de ejemplo que combina:

- Preguntas sobre la guía ***(RAG)***
- Preguntas sobre el tiempo (function calling)
- Preferencias del usuario (memoria)


In [9]:
dialogo = [
    "Hola, quiero visitar Tenerife en octubre.",
    "¿Qué zonas me recomiendas para alojarme si no quiero alquilar coche?",
    "¿Qué tiempo suele hacer en esa época?",
    "Me interesa también la gastronomía local, ¿alguna recomendación de platos típicos?",
    "¿Puedes resumirme el plan que me has propuesto hasta ahora?",
]

for q in dialogo:
    print("=" * 80)
    print("USUARIO:", q)
    r = assistant.answer(q)
    print("\nASISTENTE:\n", r)



2026-06-11 15:26:39,631 - TenerifeAssistant - INFO - Consulta recibida: 'Hola, quiero visitar Tenerife en octubre.'
2026-06-11 15:26:39,632 - src.rag - INFO - Buscando 4 chunks relevantes para: 'Hola, quiero visitar Tenerife en octubre....'


USUARIO: Hola, quiero visitar Tenerife en octubre.


2026-06-11 15:26:39,940 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:26:39,943 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:26:39,946 - TenerifeAssistant - INFO - Contexto supera 2500 tokens (3294). Se crea resumen del historial.
2026-06-11 15:26:42,793 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:26:43,871 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:26:43,877 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'user']).
2026-06-11 15:26:43,882 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant']).
2026-06-11 15:26:43,885 - TenerifeAssistant - INFO - Respuesta generada en 4.25s
2026-06-11 15:26:43,885 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué zonas me recomiendas para alojarme si no quie'
2026-06-11 15:26:43,886 - sr


ASISTENTE:
 ¡Hola! Octubre es un buen mes para visitar Tenerife, ya que el clima suele ser agradable con temperaturas templadas y menos masificación turística que en verano. ¿Te gustaría que te ayude con información sobre el clima, actividades recomendadas, o lugares que visitar en Tenerife durante ese mes?

**Fuentes:**
[1] Guía turística (pág. 16)
[2] Guía turística (pág. 15)
[3] Guía turística (pág. 11)
[4] Guía turística (pág. 22)
USUARIO: ¿Qué zonas me recomiendas para alojarme si no quiero alquilar coche?


2026-06-11 15:26:47,820 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:26:50,932 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:26:50,995 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant']).
2026-06-11 15:26:51,000 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'user']).
2026-06-11 15:26:51,004 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'assistant']).
2026-06-11 15:26:51,009 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant', 'user']).
2026-06-11 15:26:51,012 - src.memory - INFO - Historial recortado (eliminados roles: ['assistant']).
2026-06-11 15:26:51,015 - TenerifeAssistant - INFO - Respuesta generada en 7.13s
2026-06-11 15:26:51,016 - TenerifeAssistant - INFO - Consulta recibida: '¿Qué tiempo suele hacer en esa época?'
2026-06-11 15:26:51,019 - Tener

...


ASISTENTE:
 El 15 de junio de 2024 en Santa Cruz de Tenerife se espera un día con cielo claro, temperatura alrededor de 27.6 °C, humedad del 77% y viento moderado de 21.8 km/h. Es un clima ideal para disfrutar tanto de actividades al aire libre como rutas de senderismo o paseos por la ciudad.
USUARIO: Me interesa también la gastronomía local, ¿alguna recomendación de platos típicos?


2026-06-11 15:27:04,825 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-11 15:27:04,828 - src.rag - INFO - Encontrados 4 resultados relevantes
2026-06-11 15:27:04,831 - TenerifeAssistant - INFO - Contexto supera 2500 tokens (2785). Se crea resumen del historial.
2026-06-11 15:27:09,566 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:27:12,626 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:27:12,634 - TenerifeAssistant - INFO - Respuesta generada en 8.03s
2026-06-11 15:27:12,635 - TenerifeAssistant - INFO - Consulta recibida: '¿Puedes resumirme el plan que me has propuesto has'
2026-06-11 15:27:12,636 - src.rag - INFO - Buscando 4 chunks relevantes para: '¿Puedes resumirme el plan que me has propuesto has...'
2026-06-11 15:27:12,828 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1


ASISTENTE:
 En la gastronomía local de Tenerife, puedes probar varios platos típicos que reflejan la cultura canaria, entre ellos:

1. Papas arrugadas con mojo: Pequeñas papas cocidas con sal y servidas con salsas mojo rojo o verde, muy populares en Tenerife.
2. Gofio: Harina tostada de cereales que se utiliza en diversos platos, desde sopas hasta postres.
3. Conejo en salmorejo: Conejo marinado en una salsa a base de vino, ajo y especias, típico de las Islas Canarias.
4. Puchero canario: Un guiso tradicional con carne, verduras y legumbres.
5. Queso canario: Queso de cabra, tanto fresco como curado, muchas veces acompañado con miel de palma.
6. Bienmesabe: Postre dulce hecho con almendras, miel y yema de huevo.

Estos platos podrás encontrarlos en restaurantes locales y guachinches, lugares tradicionales donde se sirve comida casera típica canaria. Si quieres, puedo ayudarte a buscar ofertas gastronómicas actuales en Tenerife. ¿Te interesa?

**Fuentes:**
[1] Guía turística (pág. 24)


2026-06-11 15:27:17,117 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:27:22,333 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-11 15:27:22,346 - src.memory - INFO - Historial recortado (eliminados roles: ['user', 'assistant']).
2026-06-11 15:27:22,349 - TenerifeAssistant - INFO - Respuesta generada en 9.71s



ASISTENTE:
 Claro, este es el resumen del plan de 5 días en Tenerife que te propuse, enfocándote en senderismo y playas tranquilas, evitando lugares muy turísticos:

Día 1: Senderismo en el Barranco de Masca. Visita el pintoresco y poco masificado pueblo de Masca.  
Día 2: Disfruta playas tranquilas en Buenavista del Norte, zonas poco concurridas.  
Día 3: Senderismo en el Parque Rural de Anaga, en lugares menos conocidos como La Jardina.  
Día 4: Explora playas y calas poco accesibles y tranquilas en el norte de Tenerife.  
Día 5: Visita cultural a pueblos tradicionales como Garachico e Icod de los Vinos, con comida local en restaurantes familiares.

Para alojamiento sin coche, recomendé Santa Cruz de Tenerife, Puerto de La Cruz o Costa Adeje por su buena conexión de transporte público.

Clima en octubre: templado y agradable, ideal para actividades al aire libre.

Gastronomía local sugerida: papas arrugadas con mojo, gofio, conejo en salmorejo, puchero canario, queso canario y bienm

## Conclusiones y limitaciones

En este notebook hemos demostrado:

- Conexión con un LLM comercial (`model_name` configurado al inicio).
- Uso de RAG sobre la guía turística de Tenerife (carga del PDF, chunking, embeddings, FAISS).
- Diálogo multiturno con memoria conversacional.
- Uso de al menos una función externa (`get_weather`) mediante function calling.

### Limitaciones

- La calidad de las respuestas depende de la guía y del modelo LLM.
- La función de clima puede estar simulada o depender de una API externa con posibles errores de red.
- No se han cubierto todos los posibles casos límite de usuario.

***Este Notebook es el test01, realizare mejoras***